# Embeddings: Complete Guide with Examples and Code

**Author notes** — A hands-on reference covering what embeddings are, how they work, major types, and practical code you can run.

---

## Table of Contents

1. [What Are Embeddings?](#1-what-are-embeddings)
2. [Why Use Embeddings?](#2-why-use-embeddings)
3. [Mathematical Foundations](#3-mathematical-foundations)
4. [Classical Word Embeddings (Word2Vec, GloVe)](#4-classical-word-embeddings)
5. [Contextual Embeddings (BERT, Transformers)](#5-contextual-embeddings)
6. [Sentence & Document Embeddings](#6-sentence--document-embeddings)
7. [Image & Multimodal Embeddings](#7-image--multimodal-embeddings)
8. [Embedding Similarity & Search](#8-embedding-similarity--search)
9. [Dimensionality Reduction & Visualization](#9-dimensionality-reduction--visualization)
10. [Vector Databases & Production Use](#10-vector-databases--production-use)
11. [Choosing the Right Embedding Model](#11-choosing-the-right-embedding-model)
12. [Common Pitfalls & Best Practices](#12-common-pitfalls--best-practices)
13. [Summary Cheat Sheet](#13-summary-cheat-sheet)

---

## 1. What Are Embeddings?

An **embedding** is a **dense numerical representation** of data (text, images, audio, users, products, etc.) as a **fixed-length vector** of real numbers in a high-dimensional space.

### Key Idea

> Similar items should have **similar vectors** (close in vector space). Dissimilar items should be **far apart**.

### Example (Intuition)

| Word / Concept | Rough 3D Vector (simplified) |
|--------------|-------------------------------|
| `king`       | `[0.9, 0.1, 0.8]`             |
| `queen`      | `[0.85, 0.15, 0.75]`          |
| `apple`      | `[0.1, 0.9, 0.2]`             |

`king` and `queen` are close; both are far from `apple`.

### Formal Definition

Given an input object \( x \), an embedding function \( f \) maps it to a vector:

$$\mathbf{e} = f(x) \in \mathbb{R}^d$$

where \( d \) is the **embedding dimension** (e.g., 128, 384, 768, 1536).

### What Gets Embedded?

| Domain        | Input                    | Output Vector        |
|---------------|--------------------------|----------------------|
| NLP           | Word, sentence, document | 384–1536 dim vector  |
| Vision        | Image, video frame       | 512–2048 dim vector  |
| Audio         | Audio clip               | 128–1024 dim vector  |
| Recommendation| User, product            | 64–256 dim vector    |
| Graph         | Node, edge               | 64–512 dim vector    |

In [ ]:
# Install dependencies (run once)
# !pip install numpy matplotlib scikit-learn gensim sentence-transformers torch torchvision --quiet

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

np.random.seed(42)
print("Libraries loaded successfully.")

### Toy Example: Manual Word Vectors

Before neural networks, you can **hand-craft** tiny embeddings to understand the concept.

In [ ]:
# Dimensions: [royalty, gender_female, food]
embeddings = {
    "king":   np.array([0.90, 0.10, 0.05]),
    "queen":  np.array([0.85, 0.80, 0.05]),
    "man":    np.array([0.20, 0.10, 0.05]),
    "woman":  np.array([0.20, 0.80, 0.05]),
    "apple":  np.array([0.05, 0.10, 0.95]),
    "banana": np.array([0.05, 0.10, 0.90]),
}

def compare(word_a, word_b):
    a, b = embeddings[word_a], embeddings[word_b]
    cos_sim = cosine_similarity(a.reshape(1, -1), b.reshape(1, -1))[0, 0]
    eucl_dist = euclidean_distances(a.reshape(1, -1), b.reshape(1, -1))[0, 0]
    print(f"{word_a:8} vs {word_b:8} | cosine similarity: {cos_sim:.3f} | euclidean distance: {eucl_dist:.3f}")

compare("king", "queen")
compare("king", "apple")
compare("apple", "banana")

# Famous vector arithmetic analogy: king - man + woman ≈ queen
result = embeddings["king"] - embeddings["man"] + embeddings["woman"]
print("\nking - man + woman ≈", result)
print("Closest stored vector:", min(embeddings, key=lambda w: euclidean_distances(result.reshape(1,-1), embeddings[w].reshape(1,-1))[0,0]))

---

## 2. Why Use Embeddings?

Embeddings turn **unstructured or categorical data** into **math-friendly vectors** so machines can:

1. **Measure similarity** — "How related are these two items?"
2. **Search semantically** — Find documents by meaning, not just keywords
3. **Cluster & classify** — Group similar items; feed vectors to ML models
4. **Recommend** — "Users similar to you also liked..."
5. **Transfer learning** — Pre-trained embeddings as features for downstream tasks
6. **RAG (Retrieval-Augmented Generation)** — Retrieve relevant context for LLMs

### Embeddings vs One-Hot Encoding

| Approach        | Vector Size | Captures Meaning? | Example for "cat" |
|-----------------|-------------|-------------------|-------------------|
| One-hot         | Vocab size (huge) | No — all words equally distant | `[0,0,1,0,...,0]` |
| Embedding       | Small (e.g., 300) | Yes — similar words close | `[0.2, -0.5, 0.8, ...]` |

**One-hot** treats every word as orthogonal (zero similarity). **Embeddings** learn relationships.

In [ ]:
# One-hot vs embedding comparison
vocab = ["cat", "dog", "car", "king"]
word_to_idx = {w: i for i, w in enumerate(vocab)}

def one_hot(word):
    vec = np.zeros(len(vocab))
    vec[word_to_idx[word]] = 1
    return vec

cat_oh, dog_oh = one_hot("cat"), one_hot("dog")
print("One-hot similarity(cat, dog):", cosine_similarity(cat_oh.reshape(1,-1), dog_oh.reshape(1,-1))[0,0])
print("One-hot similarity(cat, car):", cosine_similarity(cat_oh.reshape(1,-1), one_hot("car").reshape(1,-1))[0,0])
print("→ One-hot always gives 0 similarity between different words!")

---

## 3. Mathematical Foundations

### 3.1 Dot Product

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{d} a_i b_i$$

Measures alignment of two vectors. Large positive dot product → vectors point in similar directions.

### 3.2 Cosine Similarity (Most Common for Embeddings)

$$\text{cos\_sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$$

- Range: **-1 to 1** (often 0 to 1 for NLP embeddings)
- **1** = identical direction (very similar)
- **0** = orthogonal (unrelated)
- **-1** = opposite direction

### 3.3 Euclidean Distance

$$\|\mathbf{a} - \mathbf{b}\|_2 = \sqrt{\sum_{i=1}^{d} (a_i - b_i)^2}$$

Lower distance = more similar.

### 3.4 L2 Normalization

Many embedding APIs return **unit-normalized** vectors (length = 1). After normalization:

$$\text{cos\_sim}(\mathbf{a}, \mathbf{b}) = \mathbf{a} \cdot \mathbf{b}$$

Dot product alone is enough — faster for large-scale search.

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([2.0, 4.0, 6.0])  # same direction as a
c = np.array([3.0, -1.0, 0.0])  # different direction

def l2_normalize(v):
    return v / np.linalg.norm(v)

a_norm, b_norm, c_norm = l2_normalize(a), l2_normalize(b), l2_normalize(c)

print("Cosine sim (a, b):", cosine_similarity(a.reshape(1,-1), b.reshape(1,-1))[0,0])
print("Cosine sim (a, c):", cosine_similarity(a.reshape(1,-1), c.reshape(1,-1))[0,0])
print("Dot product after L2 norm (a, b):", np.dot(a_norm, b_norm))
print("Euclidean dist (a, c):", euclidean_distances(a.reshape(1,-1), c.reshape(1,-1))[0,0])

---

## 4. Classical Word Embeddings

### 4.1 Word2Vec (2013 — Mikolov et al.)

Learns word vectors from **co-occurrence patterns** in large text corpora.

**Two architectures:**

| Model | Idea | Training Task |
|-------|------|-------------|
| **CBOW** (Continuous Bag of Words) | Predict word from context | Given surrounding words → predict center word |
| **Skip-gram** | Predict context from word | Given center word → predict surrounding words |

**Properties learned:**
- Semantic similarity: `happy` ≈ `joyful`
- Analogies: `king - man + woman ≈ queen`
- **Static embeddings**: one vector per word regardless of context

**Limitation:** "bank" (river) and "bank" (financial) share ONE vector.

In [ ]:
# Word2Vec with Gensim — train on a tiny corpus for demonstration
from gensim.models import Word2Vec

sentences = [
    ["king", "rules", "the", "kingdom", "with", "power"],
    ["queen", "rules", "the", "kingdom", "with", "wisdom"],
    ["prince", "is", "the", "son", "of", "king"],
    ["princess", "is", "the", "daughter", "of", "queen"],
    ["man", "works", "in", "the", "field"],
    ["woman", "works", "in", "the", "office"],
    ["apple", "is", "a", "fruit"],
    ["banana", "is", "a", "fruit"],
    ["car", "is", "a", "vehicle"],
    ["truck", "is", "a", "vehicle"],
]

# Note: real models need millions of sentences; this is illustrative
w2v_model = Word2Vec(sentences, vector_size=50, window=3, min_count=1, workers=1, epochs=100, seed=42)

def show_similar(model, word, topn=5):
    if word in model.wv:
        print(f"Words similar to '{word}':")
        for w, score in model.wv.most_similar(word, topn=topn):
            print(f"  {w:12} {score:.4f}")
    else:
        print(f"'{word}' not in vocabulary")

show_similar(w2v_model, "king")
show_similar(w2v_model, "apple")

# Vector arithmetic
if all(w in w2v_model.wv for w in ["king", "man", "woman"]):
    result = w2v_model.wv["king"] - w2v_model.wv["man"] + w2v_model.wv["woman"]
    nearest = w2v_model.wv.similar_by_vector(result, topn=3)
    print("\nking - man + woman →", nearest)

### 4.2 GloVe (Global Vectors for Word Representation)

**Idea:** Combine global co-occurrence statistics with local context window learning.

- Builds a **word co-occurrence matrix** \( X \) where \( X_{ij} \) = how often word \( j \) appears in context of word \( i \)
- Factorizes this matrix into word vectors
- Often performs well on **analogy** and **word similarity** benchmarks

**Word2Vec vs GloVe:**

| Aspect | Word2Vec | GloVe |
|--------|----------|-------|
| Training | Local context windows | Global co-occurrence matrix |
| Speed | Fast on large corpora | Precompute matrix, then train |
| Type | Static word embedding | Static word embedding |

### 4.3 FastText (Facebook)

Extension of Word2Vec that embeds **character n-grams**.

- Handles **out-of-vocabulary (OOV)** words better
- Good for morphologically rich languages
- Example: embedding for "unhappiness" uses subwords `un`, `happi`, `ness`

In [ ]:
# Load pre-trained GloVe-style vectors via Gensim (internet required)
# Alternative: download from https://nlp.stanford.edu/projects/glove/

try:
    import gensim.downloader as api
    print("Loading lightweight pre-trained model (glove-wiki-gigaword-50)...")
    glove = api.load("glove-wiki-gigaword-50")
    
    pairs = [("king", "queen"), ("king", "apple"), ("happy", "sad"), ("computer", "keyboard")]
    print("\nPre-trained GloVe similarities:")
    for w1, w2 in pairs:
        if w1 in glove and w2 in glove:
            sim = glove.similarity(w1, w2)
            print(f"  {w1:10} ↔ {w2:10} : {sim:.4f}")
        else:
            print(f"  {w1} or {w2} not in vocab")
except Exception as e:
    print(f"Could not load pre-trained GloVe: {e}")
    print("Run with internet access, or download vectors manually.")

---

## 5. Contextual Embeddings (Transformers)

### The Problem with Static Embeddings

The word **"bank"** in:
- "I sat by the **river bank**" → geographic
- "I went to the **bank** to deposit money" → financial

Static embeddings assign **one vector** to "bank". Contextual models assign **different vectors per usage**.

### BERT (Bidirectional Encoder Representations from Transformers)

- Reads full sentence **bidirectionally** (left + right context)
- Output: a vector per **token** (subword), not just per word
- `[CLS]` token embedding often used as **sentence-level** representation
- Dimensions: typically **768** (BERT-base) or **1024** (BERT-large)

### How It Works (Simplified)

```
Input:  [CLS] The cat sat on the mat [SEP]
         ↓       ↓   ↓   ↓  ↓  ↓  ↓    ↓
Output: 768-dim vector for EACH token (context-dependent)
```

### Other Transformer Embedding Models

| Model | Provider | Typical Use |
|-------|----------|-------------|
| `text-embedding-3-small/large` | OpenAI | General-purpose API embeddings |
| `embed-v3` / Cohere | Cohere | Search, RAG |
| `all-MiniLM-L6-v2` | Sentence-Transformers | Fast local sentence embeddings |
| `text-embedding-ada-002` | OpenAI (legacy) | Older but widely deployed |

In [ ]:
# Contextual embeddings with Hugging Face Transformers
try:
    import torch
    from transformers import AutoTokenizer, AutoModel
    
    model_name = "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.eval()
    
    sentences = [
        "He deposited money at the bank.",
        "He sat by the river bank.",
    ]
    
    def get_token_embeddings(text):
        inputs = tokenizer(text, return_tensors="pt")
        with torch.no_grad():
            outputs = model(**inputs)
        # Last hidden state: (batch, seq_len, 768)
        return inputs["input_ids"][0], outputs.last_hidden_state[0]
    
    for sent in sentences:
        token_ids, hidden = get_token_embeddings(sent)
        tokens = tokenizer.convert_ids_to_tokens(token_ids)
        # Find 'bank' token index
        bank_indices = [i for i, t in enumerate(tokens) if "bank" in t]
        print(f"\nSentence: {sent}")
        print(f"Tokens: {tokens}")
        if bank_indices:
            idx = bank_indices[0]
            vec = hidden[idx].numpy()
            print(f"'bank' embedding shape: {vec.shape}, first 5 values: {vec[:5]}")
    
    # Compare the two 'bank' vectors — they should differ!
    _, h1 = get_token_embeddings(sentences[0])
    _, h2 = get_token_embeddings(sentences[1])
    bank1 = h1[[i for i,t in enumerate(tokenizer.convert_ids_to_tokens(tokenizer(sentences[0])["input_ids"][0])) if "bank" in t][0]].numpy()
    bank2 = h2[[i for i,t in enumerate(tokenizer.convert_ids_to_tokens(tokenizer(sentences[1])["input_ids"][0])) if "bank" in t][0]].numpy()
    sim = cosine_similarity(bank1.reshape(1,-1), bank2.reshape(1,-1))[0,0]
    print(f"\nCosine similarity between two uses of 'bank': {sim:.4f}")
    print("(Lower than 1.0 proves contextual difference)")
except Exception as e:
    print(f"BERT demo skipped: {e}")
    print("Install: pip install transformers torch")

---

## 6. Sentence & Document Embeddings

For **semantic search**, **clustering**, and **RAG**, you need one vector per sentence or document.

### Common Approaches

| Method | How | Pros | Cons |
|--------|-----|------|------|
| **Mean pooling** | Average word vectors | Simple | Loses word order |
| **[CLS] token** | Use BERT's first token | Easy | Not always optimal |
| **Sentence-BERT (SBERT)** | Siamese network fine-tuned for sentences | High quality, fast | Needs model download |
| **API embeddings** | OpenAI, Cohere, Voyage, etc. | Best quality, no GPU | Cost, latency, privacy |

### Sentence-Transformers (Recommended for Local Use)

Models like `all-MiniLM-L6-v2` (384 dims) balance **speed** and **quality**.

In [ ]:
# Sentence embeddings with sentence-transformers
try:
    from sentence_transformers import SentenceTransformer
    
    st_model = SentenceTransformer("all-MiniLM-L6-v2")
    
    corpus = [
        "The cat sleeps on the sofa.",
        "A kitten is napping on the couch.",
        "Python is a programming language.",
        "Machine learning models learn from data.",
        "Neural networks are used in deep learning.",
        "I enjoy eating pizza on weekends.",
    ]
    
    query = "A cat resting on furniture"
    
    corpus_embeddings = st_model.encode(corpus, normalize_embeddings=True)
    query_embedding = st_model.encode([query], normalize_embeddings=True)[0]
    
    # Rank by cosine similarity (dot product when normalized)
    scores = corpus_embeddings @ query_embedding
    ranked = sorted(zip(corpus, scores), key=lambda x: x[1], reverse=True)
    
    print(f"Query: '{query}'\n")
    print("Ranked results:")
    for doc, score in ranked:
        print(f"  {score:.4f} | {doc}")
        
    print(f"\nEmbedding dimension: {corpus_embeddings.shape[1]}")
except Exception as e:
    print(f"Sentence-transformers demo skipped: {e}")
    print("Install: pip install sentence-transformers")

### Mean Pooling Example (DIY Sentence Embedding)

In [ ]:
# DIY sentence embedding via mean pooling of Word2Vec vectors
def sentence_embedding(words, model):
    vectors = [model.wv[w] for w in words if w in model.wv]
    if not vectors:
        return None
    return np.mean(vectors, axis=0)

s1 = sentence_embedding("king rules the kingdom".split(), w2v_model)
s2 = sentence_embedding("queen rules the kingdom".split(), w2v_model)
s3 = sentence_embedding("apple is a fruit".split(), w2v_model)

if s1 is not None:
    print("Mean-pool similarity (king sentence vs queen sentence):",
          cosine_similarity(s1.reshape(1,-1), s2.reshape(1,-1))[0,0])
    print("Mean-pool similarity (king sentence vs apple sentence):",
          cosine_similarity(s1.reshape(1,-1), s3.reshape(1,-1))[0,0])

---

## 7. Image & Multimodal Embeddings

Embeddings are not limited to text.

### Image Embeddings

| Model | Output | Use Case |
|-------|--------|----------|
| ResNet / EfficientNet | 512–2048 dim | Image classification features |
| CLIP (OpenAI) | 512 dim | Text ↔ image in **same space** |
| DINOv2 | 384–1536 dim | Self-supervised visual features |

### CLIP: The Key Multimodal Idea

CLIP trains image and text encoders so that:

> Matching (image, caption) pairs are close; non-matching pairs are far.

This enables:
- **Text-to-image search**: "a dog on the beach" → find matching images
- **Zero-shot classification**: Compare image to text labels

In [ ]:
# Image embeddings with a pre-trained ResNet (no custom training data needed)
try:
    import torch
    import torchvision.models as models
    import torchvision.transforms as transforms
    from PIL import Image
    
    # Use ResNet18 as feature extractor
    resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    resnet.fc = torch.nn.Identity()  # Remove classification head → 512-dim embedding
    resnet.eval()
    
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    def embed_image(pil_image):
        tensor = preprocess(pil_image).unsqueeze(0)
        with torch.no_grad():
            return resnet(tensor).numpy().flatten()
    
    # Create two simple synthetic images for demo
    red = Image.new("RGB", (224, 224), color=(200, 50, 50))
    blue = Image.new("RGB", (224, 224), color=(50, 50, 200))
    red2 = Image.new("RGB", (224, 224), color=(210, 60, 60))
    
    emb_red, emb_blue, emb_red2 = embed_image(red), embed_image(blue), embed_image(red2)
    
    print("ResNet18 image embedding dimension:", emb_red.shape[0])
    print("Similarity (red vs red2):", cosine_similarity(emb_red.reshape(1,-1), emb_red2.reshape(1,-1))[0,0])
    print("Similarity (red vs blue):", cosine_similarity(emb_red.reshape(1,-1), emb_blue.reshape(1,-1))[0,0])
except Exception as e:
    print(f"Image embedding demo skipped: {e}")
    print("Install: pip install torch torchvision pillow")

---

## 8. Embedding Similarity & Search

### Semantic Search Pipeline

```
1. Chunk documents → text passages
2. Embed each chunk → vector
3. Store vectors in index (vector DB or in-memory)
4. Embed user query → vector
5. Find top-k nearest vectors → return matching documents
```

### Brute-Force vs Approximate Nearest Neighbor (ANN)

| Method | Complexity | Accuracy | Scale |
|--------|------------|----------|-------|
| Brute-force | O(n·d) per query | Exact | < 100K vectors |
| HNSW, IVF | O(log n) approx | ~99% recall | Millions+ |

Popular ANN libraries: **FAISS**, **Hnswlib**, **Annoy**

In [ ]:
# Build a simple semantic search engine from scratch

documents = {
    "doc1": "Embeddings convert text into numerical vectors for machine learning.",
    "doc2": "Neural networks can learn dense representations of words and sentences.",
    "doc3": "The weather today is sunny with a chance of rain in the evening.",
    "doc4": "Vector databases enable fast similarity search at scale.",
    "doc5": "Cosine similarity measures the angle between two embedding vectors.",
    "doc6": "Basketball is a team sport played on a rectangular court.",
}

def simple_embed(text, dim=32):
    """Hash-based pseudo-embedding for demo without external models."""
    vec = np.zeros(dim)
    for token in text.lower().split():
        h = hash(token) % dim
        vec[h] += 1.0
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec

class SimpleVectorStore:
    def __init__(self):
        self.ids = []
        self.texts = []
        self.vectors = []
    
    def add(self, doc_id, text, vector):
        self.ids.append(doc_id)
        self.texts.append(text)
        self.vectors.append(vector)
    
    def search(self, query_vector, top_k=3):
        matrix = np.vstack(self.vectors)
        scores = cosine_similarity(query_vector.reshape(1, -1), matrix)[0]
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(self.ids[i], self.texts[i], scores[i]) for i in top_indices]

store = SimpleVectorStore()
for doc_id, text in documents.items():
    store.add(doc_id, text, simple_embed(text))

queries = [
    "How do word vectors work?",
    "Find information about sports",
    "Searching vectors efficiently",
]

for q in queries:
    print(f"\nQuery: {q}")
    results = store.search(simple_embed(q), top_k=3)
    for doc_id, text, score in results:
        print(f"  [{score:.3f}] {doc_id}: {text}")

In [ ]:
# FAISS example for scalable nearest-neighbor search
try:
    import faiss
    
    dim = 32
    n_vectors = 1000
    
    # Random normalized vectors (stand-in for real embeddings)
    vectors = np.random.randn(n_vectors, dim).astype("float32")
    faiss.normalize_L2(vectors)
    
    index = faiss.IndexFlatIP(dim)  # Inner product on normalized vectors = cosine sim
    index.add(vectors)
    
    query = np.random.randn(1, dim).astype("float32")
    faiss.normalize_L2(query)
    
    distances, indices = index.search(query, k=5)
    print("FAISS top-5 nearest neighbor indices:", indices[0])
    print("Similarity scores:", distances[0])
    print(f"Index contains {index.ntotal} vectors")
except ImportError:
    print("FAISS not installed. Install with: pip install faiss-cpu")

---

## 9. Dimensionality Reduction & Visualization

Embeddings live in 384–1536 dimensions — impossible to visualize directly. We project to 2D/3D.

| Method | Type | Best For |
|--------|------|----------|
| **PCA** | Linear | Fast, preserves global structure |
| **t-SNE** | Nonlinear | Clusters, exploration (misleading distances) |
| **UMAP** | Nonlinear | Balance of local + global structure |

In [ ]:
# Visualize word embeddings in 2D
words = list(embeddings.keys())
matrix = np.vstack([embeddings[w] for w in words])

pca = PCA(n_components=2)
coords_pca = pca.fit_transform(matrix)

fig, ax = plt.subplots(figsize=(8, 6))
for i, word in enumerate(words):
    ax.scatter(coords_pca[i, 0], coords_pca[i, 1], s=100)
    ax.annotate(word, (coords_pca[i, 0] + 0.02, coords_pca[i, 1] + 0.02), fontsize=12)
ax.set_title("Toy Word Embeddings — PCA Projection")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# t-SNE on Word2Vec vectors (if model trained above)
w2v_words = [w for w in w2v_model.wv.index_to_key if w.isalpha()]
if len(w2v_words) >= 5:
    w2v_matrix = np.vstack([w2v_model.wv[w] for w in w2v_words])
    
    tsne = TSNE(n_components=2, perplexity=min(5, len(w2v_words)-1), random_state=42, init="pca")
    coords_tsne = tsne.fit_transform(w2v_matrix)
    
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(coords_tsne[:, 0], coords_tsne[:, 1], alpha=0.7)
    for i, word in enumerate(w2v_words):
        ax.annotate(word, (coords_tsne[i, 0], coords_tsne[i, 1]), fontsize=9)
    ax.set_title("Word2Vec Embeddings — t-SNE (toy corpus)")
    plt.tight_layout()
plt.show()

---

## 10. Vector Databases & Production Use

### What Is a Vector Database?

A database optimized for storing and querying **high-dimensional vectors** with metadata.

| Database | Type | Notes |
|----------|------|-------|
| **Pinecone** | Managed cloud | Easy scaling, hybrid search |
| **Weaviate** | Open-source + cloud | GraphQL, hybrid BM25 + vector |
| **Milvus** | Open-source | Large scale, GPU support |
| **Chroma** | Embedded/local | Great for prototyping |
| **pgvector** | PostgreSQL extension | Vectors inside existing SQL DB |
| **Qdrant** | Open-source | Filtering, payload storage |

### Typical RAG Architecture

```
User Question
     ↓
Query Embedding (OpenAI / local model)
     ↓
Vector DB Search (top-k chunks)
     ↓
Retrieved Context + Question → LLM → Answer
```

### Metadata Filtering

Production systems often combine vector search with filters:

```python
# Pseudocode
results = vector_db.search(
    vector=query_embedding,
    top_k=10,
    filter={"department": "engineering", "year": {"$gte": 2023}}
)
```

In [ ]:
# ChromaDB quick start (local vector database)
try:
    import chromadb
    
    client = chromadb.Client()
    collection = client.create_collection("demo_embeddings")
    
    docs = [
        "Embeddings are vector representations of data.",
        "RAG combines retrieval with language model generation.",
        "Chroma is an open-source embedding database.",
    ]
    ids = ["id1", "id2", "id3"]
    
    collection.add(documents=docs, ids=ids)
    
    results = collection.query(query_texts=["What is RAG?"], n_results=2)
    print("ChromaDB query results:")
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        print(f"  distance={dist:.4f} | {doc}")
except ImportError:
    print("ChromaDB not installed. Install with: pip install chromadb")

---

## 11. Choosing the Right Embedding Model

### Decision Matrix

| Requirement | Recommended Approach |
|-------------|---------------------|
| Fast, free, local | `all-MiniLM-L6-v2` (384d) |
| Best quality, API budget | OpenAI `text-embedding-3-large` |
| Multilingual | `paraphrase-multilingual-MiniLM-L12-v2` |
| Code search | `microsoft/codebert-base` or Voyage Code |
| Long documents | Models with chunking + late interaction (ColBERT) |
| Images + text | CLIP |

### Key Evaluation Metrics (MTEB Benchmark)

- **Retrieval** — Can it find the right document?
- **Clustering** — Does it group similar items?
- **Classification** — Are vectors separable by class?
- **Semantic textual similarity (STS)** — Correlation with human judgments

### Dimension vs Performance Trade-off

| Dimensions | Storage (1M docs) | Search Speed | Quality |
|------------|-------------------|--------------|---------|
| 384 | ~1.5 GB (float32) | Fast | Good |
| 768 | ~3 GB | Medium | Better |
| 1536 | ~6 GB | Slower | Best |

In [ ]:
# Compare embedding dimensions and storage costs

num_documents = 1_000_000
dims = [384, 768, 1536]
bytes_per_float = 4  # float32

print(f"Storage estimate for {num_documents:,} documents:\n")
for d in dims:
    size_gb = (num_documents * d * bytes_per_float) / (1024**3)
    print(f"  {d:4d} dimensions → {size_gb:.2f} GB (float32)")

print("\nTip: Use float16 or int8 quantization to cut storage ~50–75% with minimal quality loss.")

---

## 12. Common Pitfalls & Best Practices

### Pitfalls

1. **Mixing embedding models** — Never embed queries with Model A and documents with Model B. Vectors are incompatible.
2. **Ignoring normalization** — Always L2-normalize if using dot product as cosine similarity.
3. **Chunk size matters** — Too small = lost context; too large = diluted meaning. Typical: 256–512 tokens.
4. **Keyword-only evaluation** — Semantic search finds paraphrases; test with varied phrasing.
5. **Static embeddings for polysemy** — Words with multiple meanings need contextual models.
6. **t-SNE distance interpretation** — Cluster proximity is meaningful; between-cluster distances are not.
7. **Domain mismatch** — General embeddings underperform on specialized domains (legal, medical).

### Best Practices

| Practice | Why |
|----------|-----|
| Use same model for index + query | Consistent vector space |
| Store raw text with vectors | Needed for display after retrieval |
| Add metadata filters | Improve precision (date, category, ACL) |
| Re-rank top results with cross-encoder | Better accuracy, higher latency |
| Monitor embedding drift | Model updates can break existing indexes |
| Fine-tune on domain data | Biggest quality gain for specialized use cases |
| Batch embed documents | API cost and throughput optimization |

In [ ]:
# Demonstrate why mixing models is dangerous

np.random.seed(0)
doc_emb_model_a = np.random.randn(384)
doc_emb_model_a /= np.linalg.norm(doc_emb_model_a)

np.random.seed(1)
query_emb_model_b = np.random.randn(384)
query_emb_model_b /= np.linalg.norm(query_emb_model_b)

np.random.seed(1)
query_emb_model_a = np.random.randn(384)
query_emb_model_a /= np.linalg.norm(query_emb_model_a)

print("Same model (correct):  ", np.dot(doc_emb_model_a, query_emb_model_a))
print("Mixed models (wrong):  ", np.dot(doc_emb_model_a, query_emb_model_b))
print("\nMixed-model similarity is meaningless — always use one model end-to-end.")

---

## 13. Summary Cheat Sheet

### Core Concepts

- **Embedding** = dense vector representing meaning/structure
- **Similarity** = cosine similarity (most common) or Euclidean distance
- **Static** (Word2Vec, GloVe) vs **Contextual** (BERT, GPT hidden states)
- **Sentence embeddings** = SBERT, mean pooling, API models

### Quick Code Patterns

```python
# 1. Encode text
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
vec = model.encode("Hello world", normalize_embeddings=True)

# 2. Cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
score = cosine_similarity(vec.reshape(1,-1), other_vec.reshape(1,-1))[0,0]

# 3. Top-k search
scores = corpus_matrix @ query_vec  # when L2-normalized
top_k = np.argsort(scores)[::-1][:5]
```

### Model Timeline

| Year | Milestone |
|------|-----------|
| 2013 | Word2Vec |
| 2014 | GloVe |
| 2016 | FastText |
| 2018 | BERT (contextual) |
| 2019 | Sentence-BERT |
| 2021 | CLIP (multimodal) |
| 2022+ | OpenAI/Cohere embedding APIs, MTEB benchmarks |

---

## Further Reading

- [Word2Vec Paper (Mikolov et al.)](https://arxiv.org/abs/1301.3781)
- [GloVe Paper (Pennington et al.)](https://nlp.stanford.edu/pubs/glove.pdf)
- [BERT Paper (Devlin et al.)](https://arxiv.org/abs/1810.04805)
- [Sentence-BERT Paper](https://arxiv.org/abs/1908.10084)
- [MTEB Leaderboard](https://huggingface.co/spaces/mteb/leaderboard)
- [FAISS Documentation](https://github.com/facebookresearch/faiss/wiki)

---

*End of notebook. Run cells top-to-bottom. Install optional packages as noted in each section for full demos.*